In [2]:
"""
=============================================================================
Project: Deep-Sea Hydrothermal Vent & Chemosynthetic Extremophile Biomass Tracker
Module: Data Science SQL Database Assessment
Description: 
    Programmatically generates a normalized SQLite database modeling deep-sea 
    submersible vent operations and extremophile biological telemetry. 
    Includes rigorous referential integrity, compound primary keys, all four 
    statistical data types, and deliberate real-world data imperfections.
=============================================================================
"""

import sqlite3
import random
import numpy as np
import os

# Define database filename
db_file = "abyssal_hydrothermal_lab.db"

# Cleanup stale file locks if executing iteratively in Jupyter
if os.path.exists(db_file):
    try:
        os.remove(db_file)
    except Exception:
        pass

# Establish connection explicitly
conn = sqlite3.connect(db_file)
cursor = conn.cursor()

# RATIONALE: Enable foreign key enforcement for strict relational integrity.
cursor.execute("PRAGMA foreign_keys = ON;")

# --- DATABASE SCHEMA ARCHITECTURE & NORMALIZATION (3NF) ---
cursor.executescript("""
DROP TABLE IF EXISTS extremophile_specimen_logs;
DROP TABLE IF EXISTS vent_chimneys;
DROP TABLE IF EXISTS submersible_dives;

-- Table 1: Submersible Dives (Parent entity storing high-level geographic mission data)
CREATE TABLE submersible_dives (
    dive_id INTEGER PRIMARY KEY AUTOINCREMENT,
    trench_sector_code TEXT NOT NULL,  -- Nominal Data: Qualitative categorical region label
    surface_salinity_ppt REAL CHECK(surface_salinity_ppt >= 0) -- Ratio Data: Absolute physical metric
);

-- Table 2: Vent Chimneys (Child entity linked to submersible_dives via Foreign Key)
CREATE TABLE vent_chimneys (
    chimney_id INTEGER PRIMARY KEY AUTOINCREMENT,
    dive_id INTEGER,
    mineral_facies_type TEXT NOT NULL,  -- Nominal Data: Qualitative categorical mineral classification
    buoyant_plume_height_m REAL CHECK(buoyant_plume_height_m >= 0), -- Ratio Data: Absolute height metric
    FOREIGN KEY (dive_id) REFERENCES submersible_dives(dive_id) ON DELETE CASCADE
);

-- Table 3: Extremophile Specimen Logs (Fact table with Compound Primary Key)
CREATE TABLE extremophile_specimen_logs (
    chimney_id INTEGER,
    specimen_id INTEGER,
    organism_classification TEXT NOT NULL,       -- Nominal Data: Qualitative categorical species tag
    metabolic_activity_tier TEXT CHECK(metabolic_activity_tier IN ('Dormant', 'Stressed', 'Active', 'Hyper_Proliferating')), -- Ordinal Data: Ordered biological progression
    plume_temperature_celsius REAL,             -- Interval Data: Quantitative scale with arbitrary zero point (Celsius)
    biomass_dry_weight_mg REAL CHECK(biomass_dry_weight_mg >= 0), -- Ratio Data: Absolute weight with true zero
    
    -- RATIONALE: Compound primary key (chimney_id + specimen_id) uniquely identifies 
    -- individual biological sample logs extracted across thermal chimneys.
    PRIMARY KEY (chimney_id, specimen_id),
    FOREIGN KEY (chimney_id) REFERENCES vent_chimneys(chimney_id) ON DELETE CASCADE
);
""")

# --- DATA POPULATION STRATEGY ---

# 1. Populate Parent Table: Submersible Dives (5 deep-sea research missions)
dives = [
    ('Mariana-Challenger-Deep', 34.5), 
    ('Kermadec-Trench-Axis', 35.1), 
    ('Mid-Atlantic-Ridge-Sector', 33.8), 
    ('Galapagos-Rift-Zone', 34.2), 
    ('East-Pacific-Rise', 34.9)
]
cursor.executemany("INSERT INTO submersible_dives (trench_sector_code, surface_salinity_ppt) VALUES (?, ?)", dives)

# 2. Populate Intermediate Table: Vent Chimneys (80 distinct hydrothermal chimneys)
chimneys_data = []
for _ in range(80):
    d_id = random.randint(1, 5)
    facies = random.choice(['Black-Smoker', 'White-Smoker', 'Serpentinite-Mound'])
    height = round(random.uniform(1.5, 45.0), 2)
    chimneys_data.append((d_id, facies, height))

cursor.executemany("INSERT INTO vent_chimneys (dive_id, mineral_facies_type, buoyant_plume_height_m) VALUES (?, ?, ?)", chimneys_data)

# 3. Populate Fact Table: Extremophile Specimen Logs (1,200+ rows)
species_list = ['Riftia_pachyptila', 'Alvinella_pompejana', 'Methanothermobacter', 'Bathymodiolus_thermophilus', 'Zoarcid_Fish']
tiers = ['Dormant', 'Stressed', 'Active', 'Hyper_Proliferating']

specimen_data = []
for row_num in range(1, 1201):
    c_id = random.randint(1, 80)
    # Composite key component ensuring valid compound combinations
    s_id = (row_num % 50) + 1  
    species = random.choice(species_list)
    tier = random.choice(tiers)
    
    # RATIONALE FOR REALISM & MISSING DATA:
    # Deep-sea sensor probes experience severe pressure interference and thermal shocks. 
    # We deliberately inject NULL values in ~4% of rows to simulate authentic equipment telemetry dropouts.
    temp = round(random.uniform(2.0, 385.0), 2) if row_num % 25 != 0 else None
    
    # RATIONALE FOR STATISTICAL DISTRIBUTION:
    # Biomass distribution follows an exponential decay curve (numerous microscopic microbes, rare massive mega-fauna).
    mass = round(np.random.exponential(scale=2.5), 4) if row_num % 30 != 0 else None
    
    specimen_data.append((c_id, s_id, species, tier, temp, mass))

# RATIONALE FOR DUPLICATE DATA INJECTION:
# Inject 35 duplicate composite records to simulate acoustic transmission echoes and operator logging overlaps.
for _ in range(35):
    specimen_data.append(random.choice(specimen_data))

# Insert data in bulk using INSERT OR IGNORE to handle duplicate constraint checks safely
cursor.executemany("""
    INSERT OR IGNORE INTO extremophile_specimen_logs (chimney_id, specimen_id, organism_classification, metabolic_activity_tier, plume_temperature_celsius, biomass_dry_weight_mg)
    VALUES (?, ?, ?, ?, ?, ?)
""", specimen_data)

conn.commit()
conn.close()

print("Deep-Sea Abyssal Hydrothermal Database successfully generated!")

Deep-Sea Abyssal Hydrothermal Database successfully generated!
